# LGD Model - Loss Given Default

Since the synthetic dataset does not include actual recovery/collateral data, this notebook estimates LGD using segment-based assumptions grounded in typical real-world retail lending patterns: secured loans (e.g. auto) recover more via collateral than unsecured loans (e.g. personal), and higher-balance loans are assumed to have marginally better recovery due to more aggressive collection efforts. This is a simplification, disclosed transparently, standing in for a full recovery-rate regression model that would require historical default/recovery data not available here.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_loan_data.csv')
df.shape

(10000, 17)

In [2]:
base_recovery_rate = {
    'auto': 0.55,
    'home_improvement': 0.35,
    'debt_consolidation': 0.15,
    'personal': 0.10,
}

df['base_recovery_rate'] = df['loan_purpose'].map(base_recovery_rate)

In [3]:
balance_percentile = df['current_balance'].rank(pct=True)
balance_adjustment = (balance_percentile - 0.5) * 0.10  # +/- up to 5 percentage points

df['adjusted_recovery_rate'] = (df['base_recovery_rate'] + balance_adjustment).clip(0.02, 0.90)

In [4]:
df['LGD'] = 1 - df['adjusted_recovery_rate']
df['LGD'] = df['LGD'].round(3)

df[['loan_id', 'loan_purpose', 'current_balance', 'adjusted_recovery_rate', 'LGD']].head(10)

,loan_id,loan_purpose,current_balance,adjusted_recovery_rate,LGD
0,1,personal,7697.92,0.10766,0.892
1,2,personal,17050.19,0.13665,0.863
2,3,auto,14505.45,0.58146,0.419
3,4,auto,4821.45,0.54120,0.459
4,5,personal,3186.66,0.07924,0.921
5,6,home_improvement,16072.24,0.38492,0.615
6,7,home_improvement,1414.64,0.31273,0.687
7,8,home_improvement,14163.25,0.38067,0.619
8,9,debt_consolidation,1454.52,0.11328,0.887
9,10,auto,12678.60,0.57675,0.423


In [5]:
df.groupby('loan_purpose')['LGD'].mean().sort_values()

loan_purpose
auto                  0.450180
home_improvement      0.650019
debt_consolidation    0.849052
personal              0.900753
Name: LGD, dtype: float64

In [6]:
df[['loan_id', 'LGD', 'adjusted_recovery_rate']].to_csv('../data/processed/lgd_estimates.csv', index=False)
print("Saved.")

Saved.
